In [2]:
from flask import Flask, request, jsonify
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image
import torch
import io

In [3]:
app = Flask(__name__)

# Load pretrained model and processor once at startup
model_name = "llava-hf/llava-1.5-7b-hf"
processor = AutoProcessor.from_pretrained(model_name)
model = LlavaForConditionalGeneration.from_pretrained(model_name)
model.eval()  # Set model to evaluation mode

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

LlavaForConditionalGeneration(
  (model): LlavaModel(
    (vision_tower): CLIPVisionModel(
      (vision_model): CLIPVisionTransformer(
        (embeddings): CLIPVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (position_embedding): Embedding(577, 1024)
        )
        (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-23): 24 x CLIPEncoderLayer(
              (self_attn): CLIPAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
              )
              (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
       

In [4]:
def generate_caption(image: Image.Image, prompt: str = "Describe this image in detail.") -> str:
    # Prepare inputs
    inputs = processor(prompt, images=image, return_tensors="pt")

    # If using GPU, move inputs to device
    # inputs = {k: v.to(device) for k,v in inputs.items()}

    # Generate output ids
    generate_ids = model.generate(**inputs, max_new_tokens=150)
    # Decode to string
    caption = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return caption

In [5]:
@app.route("/caption", methods=["POST"])
def caption():
    if "image" not in request.files:
        return jsonify({"error": "No image file provided. Use form-data key 'image'."}), 400

    img_file = request.files["image"]
    prompt = request.form.get("prompt", "Describe this image in detail.")

    # Read image via PIL
    try:
        image = Image.open(img_file.stream).convert("RGB")
    except Exception as e:
        return jsonify({"error": f"Invalid image file: {str(e)}"}), 400

    # Generate caption
    try:
        caption_text = generate_caption(image, prompt)
    except Exception as e:
        return jsonify({"error": f"Model inference failed: {str(e)}"}), 500

    return jsonify({"caption": caption_text})


In [6]:
if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.0.0.146:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
